# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

# 1) Data Contract

## My Data Lane: Search Performance Analysis

### What does one row mean?

One row represents one search performance record for a specific query, page, date, and search context.

### Which tables will I use?

I will use the Search Console warehouse table because it contains query performance data such as clicks, impressions, and ranking information.

### What time window will I use?

I will use month 2026-03 because it is a middle-panel month and avoids using the final month as development data.

### What will I predict?

I will predict search performance using clicks as a proxy label.

### What will I exclude?

I will exclude future information such as future clicks or future rankings to avoid data leakage.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Fields: Feature / Label / Context / Excluded

### Features
The features used for this lane are historical search performance and content-related fields, such as impressions, previous clicks, ranking position, query information, and content attributes. These features are available before the prediction moment and can be used as model inputs.

### Label (Target / Proxy)
The label for this task is click performance. Clicks are used as a proxy to measure search result performance and represent the outcome that the model would predict or rank.

### Context
Context fields include information such as date/month, device type, country, and content category. These fields describe the environment in which the search interaction happened and help provide additional meaning to the prediction.

### Excluded
I deliberately exclude future information such as future clicks, future rankings, and any post-outcome metrics. These fields would not be available at the decision moment and could cause data leakage, resulting in unrealistic model performance.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [33]:
import os

os.listdir()

['.config', 'content_refresh_anonymized.csv', 'sample_data']

In [34]:
from google.colab import files

uploaded = files.upload()

Saving content_refresh_anonymized.csv to content_refresh_anonymized (1).csv


In [35]:
import os

os.listdir()

['.config',
 'content_refresh_anonymized.csv',
 'content_refresh_anonymized (1).csv',
 'sample_data']

In [36]:
import os

os.listdir()

['.config',
 'content_refresh_anonymized.csv',
 'content_refresh_anonymized (1).csv',
 'sample_data']

In [37]:
import pandas as pd

df = pd.read_csv("content_refresh_anonymized.csv")

In [38]:
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [39]:
df.shape

(30000, 44)

In [40]:
df.columns.tolist()

['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

In [41]:
len(df)

30000

In [42]:
df.duplicated(subset=["content_id"]).sum()

np.int64(0)

In [43]:
df["ctr"].notna().sum()

np.int64(30000)

In [44]:
df[["ctr","avg_position","scroll_rate"]].notna().sum()

,0
ctr,30000
avg_position,30000
scroll_rate,29875


In [45]:
features = df[[
    "ctr",
    "avg_position",
    "scroll_rate",
    "word_count",
    "ai_traffic_pct"
]]

features.head()

,ctr,avg_position,scroll_rate,word_count,ai_traffic_pct
0,0.76,10.6,4.55,3221.0,0.0
1,0.05,20.3,10.00,2481.0,0.0
2,0.09,36.5,28.57,3515.0,0.0
3,0.49,6.2,3.45,NaN,0.0
4,0.13,44.0,24.29,2803.0,0.0


In [46]:
print("Total rows:", len(df))

Total rows: 30000


In [47]:
print("Duplicate content_id rows:", df.duplicated(subset=["content_id"]).sum())

Duplicate content_id rows: 0


In [48]:
print(df[["ctr","avg_position","scroll_rate","word_count","ai_traffic_pct"]].notna().sum())

ctr               30000
avg_position      30000
scroll_rate       29875
word_count        22301
ai_traffic_pct    30000
dtype: int64


In [49]:
honest_features = df[
    ["ctr", "avg_position", "scroll_rate", "word_count", "ai_traffic_pct"]
]

leaky_features = df[
    ["ctr", "avg_position", "scroll_rate", "word_count", "ai_traffic_pct", "trend_pct"]
]

print("Honest features:")
print(honest_features.head())

print("\nLeaky features:")
print(leaky_features.head())

Honest features:
    ctr  avg_position  scroll_rate  word_count  ai_traffic_pct
0  0.76          10.6         4.55      3221.0             0.0
1  0.05          20.3        10.00      2481.0             0.0
2  0.09          36.5        28.57      3515.0             0.0
3  0.49           6.2         3.45         NaN             0.0
4  0.13          44.0        24.29      2803.0             0.0

Leaky features:
    ctr  avg_position  scroll_rate  word_count  ai_traffic_pct  trend_pct
0  0.76          10.6         4.55      3221.0             0.0      -41.4
1  0.05          20.3        10.00      2481.0             0.0      -57.7
2  0.09          36.5        28.57      3515.0             0.0      -60.9
3  0.49           6.2         3.45         NaN             0.0      -13.8
4  0.13          44.0        24.29      2803.0             0.0      -34.7


I intentionally added trend_pct as a feature. This is data leakage because trend_pct is derived from the target (trend_direction). A model using this feature would achieve unrealistically high performance. I removed trend_pct and kept only the honest features.

Limitation: This analysis uses the anonymized starter dataset (30,000 rows) instead of the complete production warehouse, so the results may not fully represent production data.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The dataset contains 30,000 anonymized content items instead of the full production warehouse.
Some features have missing values (for example, word_count).
IDs (content_id, client_id) are used only for identification and not as model features

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.